# Security exploit-attempt notebook (local-only)

This notebook demonstrates common exploit attempts against `DataGateway`, `DataHelper`, and datacube-backed loads, and records which are handled by default vs not handled.

All examples run against a temporary local SQLite database and an in-memory datacube loader.

In [1]:
from __future__ import annotations

import tempfile
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

from boti_data import DataGateway, DataHelper, DatacubeConfig, DatacubeContract, SqlDatabaseConfig

tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "security_exploit_demo.db"

engine = create_engine(f"sqlite:///{db_path}")
with engine.begin() as conn:
    conn.execute(text("CREATE TABLE users (id INTEGER PRIMARY KEY, status TEXT)"))
    conn.execute(text("INSERT INTO users (status) VALUES ('active'), ('inactive')"))
engine.dispose()

sql_config = SqlDatabaseConfig(
    connection_url=f"sqlite:///{db_path}",
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
)

def user_count() -> int:
    check_engine = create_engine(f"sqlite:///{db_path}")
    try:
        with check_engine.connect() as conn:
            return int(conn.execute(text("SELECT COUNT(*) FROM users")).scalar_one())
    finally:
        check_engine.dispose()

results: list[dict[str, object]] = []

def record(component: str, case: str, expected: str, observed: str, handled: bool, notes: str) -> None:
    results.append(
        {
            "component": component,
            "case": case,
            "expected": expected,
            "observed": observed,
            "handled": handled,
            "notes": notes,
        }
    )

## DataGateway exploit attempts

In [2]:
# 1) Mutating raw SQL is blocked.
try:
    with DataGateway(sql_config) as gateway:
        gateway.load(sql="DELETE FROM users", as_pandas=True, allow_raw_sql=True)
    record("DataGateway", "mutating raw sql", "blocked", "allowed", False, "Unexpected: DELETE should be blocked.")
except ValueError as exc:
    record("DataGateway", "mutating raw sql", "blocked", "blocked", True, str(exc))

# 2) Multi-statement payload is blocked.
try:
    with DataGateway(sql_config) as gateway:
        gateway.load(sql="SELECT 1 AS id; DROP TABLE users", as_pandas=True, allow_raw_sql=True)
    record("DataGateway", "multi statement sql", "blocked", "allowed", False, "Unexpected: multi-statement SQL should be blocked.")
except ValueError as exc:
    record("DataGateway", "multi statement sql", "blocked", "blocked", True, str(exc))

# 3) Parameterized SQL injection payload is handled (payload treated as value).
payload = "active'; DROP TABLE users; --"
with DataGateway(sql_config) as gateway:
    frame = gateway.load(
        sql="SELECT id, status FROM users WHERE status = :status",
        params={"status": payload},
        as_pandas=True,
        allow_raw_sql=True,
    )
count_after_param_test = user_count()
handled_param = bool(frame.empty and count_after_param_test == 2)
record(
    "DataGateway",
    "parameterized injection payload",
    "handled",
    "handled" if handled_param else "not handled",
    handled_param,
    "Table unchanged and payload returned no rows.",
)

# 4) Unsafe string interpolation is NOT handled by gateway guardrails.
tautology_payload = "active' OR 1=1 --"
unsafe_sql = f"SELECT id, status FROM users WHERE status = '{tautology_payload}'"
with DataGateway(sql_config) as gateway:
    unsafe_frame = gateway.load(sql=unsafe_sql, as_pandas=True, allow_raw_sql=True)
not_handled = len(unsafe_frame) > 1
record(
    "DataGateway",
    "unsafe interpolated raw sql",
    "blocked or neutralized",
    "not handled" if not_handled else "handled",
    not not_handled,
    "Guardrails cannot fix unsafe app-side SQL string interpolation. Use bound params.",
)

## DataHelper and Datacube attempts

In [3]:
# DataHelper uses the same gateway protections.
try:
    with DataHelper(sql_config, raw_sql_policy="disabled") as helper:
        helper.load(sql="SELECT 1 AS id", as_pandas=True, allow_raw_sql=True)
    record("DataHelper", "raw sql policy disabled", "blocked", "allowed", False, "Unexpected: policy should block raw SQL.")
except ValueError as exc:
    record("DataHelper", "raw sql policy disabled", "blocked", "blocked", True, str(exc))

with DataHelper(sql_config) as helper:
    helper_unsafe = helper.load(sql=unsafe_sql, as_pandas=True, allow_raw_sql=True)
helper_not_handled = len(helper_unsafe) > 1
record(
    "DataHelper",
    "unsafe interpolated raw sql",
    "blocked or neutralized",
    "not handled" if helper_not_handled else "handled",
    not helper_not_handled,
    "Same limitation as DataGateway: use params instead of interpolation.",
)

# Datacube: permissive contract accepts suspicious inputs by default.
def permissive_loader(request):
    return pd.DataFrame(
        [{"cube": request.cube, "filter_keys": sorted([str(k) for k in request.filters.keys()])}]
    )

with DataGateway(DatacubeConfig(loader=permissive_loader, default_cube="sales")) as cube_gateway:
    permissive_frame = cube_gateway.load(
        cube="../admin",
        filters={"$where": "1=1"},
        return_type="pandas",
    )
permissive_allowed = not permissive_frame.empty
record(
    "Datacube",
    "suspicious cube/filter without validator",
    "blocked",
    "not handled" if permissive_allowed else "blocked",
    not permissive_allowed,
    "Datacube security is contract-defined; add request_validator.",
)

# Datacube: strict validator blocks suspicious request.
def strict_validator(request):
    cube_name = str(request.cube or "")
    if ".." in cube_name or "/" in cube_name:
        raise ValueError("cube name contains forbidden path tokens")
    if any(str(key).startswith("$") for key in request.filters):
        raise ValueError("operator-style filter keys are not allowed")

strict_contract = DatacubeContract(request_validator=strict_validator)

try:
    with DataGateway(
        DatacubeConfig(loader=permissive_loader, contract=strict_contract, default_cube="sales")
    ) as cube_gateway:
        cube_gateway.load(cube="../admin", filters={"$where": "1=1"}, return_type="pandas")
    record("Datacube", "suspicious cube/filter with validator", "blocked", "allowed", False, "Unexpected: validator should reject request.")
except ValueError as exc:
    record("Datacube", "suspicious cube/filter with validator", "blocked", "blocked", True, str(exc))

In [4]:
report = pd.DataFrame(results)
report = report[["component", "case", "expected", "observed", "handled", "notes"]]
report

,component,case,expected,observed,handled,notes
0,DataGateway,mutating raw sql,blocked,blocked,True,1 validation error for SqlLoadRequest\n Value...
1,DataGateway,multi statement sql,blocked,blocked,True,1 validation error for SqlLoadRequest\n Value...
2,DataGateway,parameterized injection payload,handled,handled,True,Table unchanged and payload returned no rows.
3,DataGateway,unsafe interpolated raw sql,blocked or neutralized,not handled,False,Guardrails cannot fix unsafe app-side SQL stri...
4,DataHelper,raw sql policy disabled,blocked,blocked,True,Raw sql= execution is disabled by this DataGat...
5,DataHelper,unsafe interpolated raw sql,blocked or neutralized,not handled,False,Same limitation as DataGateway: use params ins...
6,Datacube,suspicious cube/filter without validator,blocked,not handled,False,Datacube security is contract-defined; add req...
7,Datacube,suspicious cube/filter with validator,blocked,blocked,True,Datacube contract request validation failed: c...


In [5]:
tmp_dir.cleanup()
print("Temporary resources cleaned up.")

Temporary resources cleaned up.


In [6]:
report.to_json()

'{"component":{"0":"DataGateway","1":"DataGateway","2":"DataGateway","3":"DataGateway","4":"DataHelper","5":"DataHelper","6":"Datacube","7":"Datacube"},"case":{"0":"mutating raw sql","1":"multi statement sql","2":"parameterized injection payload","3":"unsafe interpolated raw sql","4":"raw sql policy disabled","5":"unsafe interpolated raw sql","6":"suspicious cube\\/filter without validator","7":"suspicious cube\\/filter with validator"},"expected":{"0":"blocked","1":"blocked","2":"handled","3":"blocked or neutralized","4":"blocked","5":"blocked or neutralized","6":"blocked","7":"blocked"},"observed":{"0":"blocked","1":"blocked","2":"handled","3":"not handled","4":"blocked","5":"not handled","6":"not handled","7":"blocked"},"handled":{"0":true,"1":true,"2":true,"3":false,"4":true,"5":false,"6":false,"7":true},"notes":{"0":"1 validation error for SqlLoadRequest\\n  Value error, Raw sql= only supports single-statement read-only SELECT\\/WITH queries. Mutating or multi-statement SQL is blo